In [6]:
import openpyxl

def load_retail_data(filepath, max_rows=50000):
    """Reads the retail Excel file. max_rows limits rows for speed."""
    wb  = openpyxl.load_workbook(filepath, read_only=True)
    ws  = wb.active
    rows = []
    headers = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i == 0:
            headers = row   # first row is column names
            continue
        if i > max_rows:
            break
        # Pack each row into a dictionary using headers as keys
        row_dict = dict(zip(headers, row))
        rows.append(row_dict)
    wb.close()
    return rows

data = load_retail_data('Documents/AI Engineer Learning/data/online_retail_II.xlsx')
print(f"Loaded {len(data):,} rows")

Loaded 50,000 rows


In [7]:
def aggregate_by_country(rows):
    """Groups rows by country. Returns dict: {country: {revenue, orders}}."""
    summary = {}
    skipped = 0
    for row in rows:
        invoice  = str(row.get("Invoice", "") or "")
        quantity = row.get("Quantity") or 0
        price    = row.get("Price")    or 0
        country  = row.get("Country")  or "Unknown"

        # Skip cancelled orders (Invoice starts with 'C') or negative qty
        if invoice.startswith("C") or quantity <= 0:
            skipped += 1
            continue

        revenue = quantity * price
        if country not in summary:
            summary[country] = {"revenue": 0, "orders": 0}
        summary[country]["revenue"] += revenue
        summary[country]["orders"]  += 1

    print(f"Skipped {skipped:,} cancelled/invalid rows")
    return summary

country_data = aggregate_by_country(data)
print(f"Found {len(country_data)} unique countries")

Skipped 1,008 cancelled/invalid rows
Found 23 unique countries


In [8]:
def classify_country(revenue):
    if revenue >= 100000: return "🔥 High"
    elif revenue >= 20000: return "✅ Medium"
    else:                  return "⚠️  Low"

def save_results(country_data, filename="retail_results.txt"):
    # Sort countries by revenue, highest first
    sorted_data = sorted(country_data.items(),
                          key=lambda x: x[1]["revenue"], reverse=True)
    with open(filename, "w", encoding="utf-8") as f:
        f.write("RETAIL REVENUE ANALYSIS BY COUNTRY\n")
        f.write("=" * 50 + "\n\n")
        for country, stats in sorted_data:
            tier    = classify_country(stats["revenue"])
            revenue = stats["revenue"]
            orders  = stats["orders"]
            line    = f"{country:<20} Rev: £{revenue:>12,.2f}  Orders: {orders:>5}  {tier}\n"
            f.write(line)
    print(f"Results saved to {filename}")

save_results(country_data)
print("Pipeline complete!")

Results saved to retail_results.txt
Pipeline complete!
